# 🦀 Day 2: Tokio Runtime

Today: `#[tokio::main]`, spawning tasks, `JoinHandle`, and async sleep.

---

## Tokio — The Async Runtime

Tokio is the most popular async runtime for Rust. It provides:
- Task scheduling (many tasks on few threads)
- Async I/O (network, files)
- Timers and timeouts

In EvCxR we can't use `#[tokio::main]`, so we use `Runtime::new().unwrap().block_on(...)` instead.

In [ ]:
:dep tokio = { version = "1", features = ["full"] }

In [ ]:
use tokio::runtime::Runtime;
use tokio::time::{sleep, Duration};

// tokio::spawn runs a future concurrently — returns JoinHandle
let rt = Runtime::new().unwrap();
rt.block_on(async {
    let handle = tokio::spawn(async {
        sleep(Duration::from_millis(100)).await;
        "Task finished!"
    });
    let result = handle.await.unwrap();
    println!("{}", result);
});

## JoinHandle and .await

- `tokio::spawn(future)` returns a `JoinHandle<T>`
- `.await` on the handle waits for the task to complete
- Returns `Result<T, JoinError>` — unwrap or handle the error

In [ ]:
// Running multiple tasks concurrently
let rt = Runtime::new().unwrap();
rt.block_on(async {
    let h1 = tokio::spawn(async {
        sleep(Duration::from_millis(80)).await;
        1
    });
    let h2 = tokio::spawn(async {
        sleep(Duration::from_millis(50)).await;
        2
    });
    let h3 = tokio::spawn(async {
        sleep(Duration::from_millis(30)).await;
        3
    });
    let (a, b, c) = (h1.await.unwrap(), h2.await.unwrap(), h3.await.unwrap());
    println!("Results: {}, {}, {} — ran concurrently!", a, b, c);
});

## Task vs Thread

| | Thread | Task |
|---|---|---|
| Unit | OS thread | Lightweight async task |
| Cost | ~1–2 MB stack | Small heap allocation |
| Count | Thousands max | Millions possible |
| Scheduling | OS | Tokio runtime |

In [ ]:
// tokio::time::sleep vs std::thread::sleep
// thread::sleep BLOCKS the thread — bad in async!
// tokio::time::sleep YIELDS — lets other tasks run

let rt = Runtime::new().unwrap();
rt.block_on(async {
    let start = std::time::Instant::now();
    let h1 = tokio::spawn(async {
        sleep(Duration::from_millis(50)).await;
        "A"
    });
    let h2 = tokio::spawn(async {
        sleep(Duration::from_millis(50)).await;
        "B"
    });
    let (a, b) = (h1.await.unwrap(), h2.await.unwrap());
    println!("{}, {} — took ~50ms (concurrent), not 100ms", a, b);
    println!("Elapsed: {:?}", start.elapsed());
});

## 🏋️ Exercise

In [ ]:
// Exercise: Spawn 4 tasks that each compute a value (e.g. square of 1, 2, 3, 4).
// Run them concurrently, collect all results, and print the sum.

// YOUR CODE HERE

## 🎯 Key Takeaways

1. **Tokio** is the dominant async runtime for Rust
2. Use `Runtime::block_on(async { ... })` in EvCxR (no `#[tokio::main]`)
3. `tokio::spawn(future)` runs a task concurrently, returns `JoinHandle`
4. `.await` on `JoinHandle` waits for the task and gets its result
5. Use `tokio::time::sleep` in async code — never `thread::sleep`!

---
**Tomorrow:** Async I/O — files and network with Tokio! 📂🌐